# Stage-2 BO campaign — T2 final report (prompt 30, scenario C, q-ParEGO)

Executed record behind `BO_STAGE2.md`. Data: the 32 in-box starting points
(`stage2_C_n0` + `stage2_C_n0b`) + `waves/bo_C_r1` + `waves/bo_C_r2`, the
ledger, `round_records.json`. All GP/HV code is imported from
`acquire_next.py` (the same functions the CRC bot ran) and bo-gp
`feat/qparego` @ `60e6328`. M = {load_shed_mwh, true_curtailment_mwh,
total_cost_less_synthetic_usd}, all minimized. **Costs carry the
pending-g1 caveat** (constant +$28.188M add-back under option (c) — ranks
and dominance invariant).

In [1]:
import os, sys, json, itertools
os.environ.setdefault("BOGP_PATH", "/Users/yilu/Documents/GitHub/bo-gp-qparego")
CAMP = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, torch, gpytorch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import acquire_next as an
from bogp.mobo.pareto import hypervolume_exact
import subprocess
print("bo-gp HEAD:", subprocess.run(["git", "-C", os.environ["BOGP_PATH"], "rev-parse", "HEAD"],
                                    capture_output=True, text=True).stdout.strip())
FIG = os.path.join(os.getcwd(), "figs"); os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.labelweight": "bold", "axes.titleweight": "bold",
                     "axes.linewidth": 1.6, "xtick.major.width": 1.4, "ytick.major.width": 1.4,
                     "font.weight": "bold", "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.dpi": 200, "savefig.bbox": "tight"})
C_START, C_R1, C_R2, C_FRONT = "#8a8985", "#2a78d6", "#eb6834", "#1baf7a"
M, T = an.M_COLS, an.SORTED_TIERS
SHORT = {"load_shed_mwh": "Load shed [MWh]", "true_curtailment_mwh": "Curtailment [MWh]",
         "total_cost_less_synthetic_usd": "Cost less synthetic [$M]"}
SHORT_K = {"load_shed_mwh": "Load shed [GWh]", "true_curtailment_mwh": "Curtailment [GWh]",
           "total_cost_less_synthetic_usd": "Cost less synthetic [$M]"}
PMAX = {"nuclear": 400.0, "pv": 188.2 + 49.7 + 51.6 + 51.0, "tail": 51.6 + 51.6 + 148.3,
        "wind_122": 713.5, "wind_303": 847.0, "wind_317": 799.1}   # gen.csv PMax sums
led = an.load_ledger(CAMP)
recs = json.load(open(an.records_path(CAMP)))
REF = np.asarray(led["ref_point"], float)

bo-gp HEAD: 60e6328236e745b3d34dc8cd704bb72888ae5d15


## 0. Data

In [2]:
frames = []
for w, src in [("stage2_C_n0", "start"), ("stage2_C_n0b", "start"), ("bo_C_r1", "r1"), ("bo_C_r2", "r2")]:
    dm = pd.read_csv(os.path.join(CAMP, "waves", w, "design_matrix.csv"))
    ob = pd.read_csv(os.path.join(CAMP, "waves", w, "objectives.csv"))
    d = dm.merge(ob[["index"] + M], on="index", validate="1:1")
    d["wave"], d["src"] = w, src
    frames.append(d)
D = pd.concat(frames, ignore_index=True)
D["label"] = D["wave"].str.replace("stage2_C_", "").str.replace("bo_C_", "") + "#" + D["index"].astype(str)
for t in T:
    D[f"{t}_mw"] = D[f"{t}_omega"] * PMAX[t]
D["total_mw"] = D[[f"{t}_mw" for t in T]].sum(axis=1)
X_all = D[[f"{t}_omega" for t in T]].to_numpy(float); Y_all = D[M].to_numpy(float)
print(D.groupby("src").size().to_dict(), "n =", len(D))
X2, Y2 = an.load_training(CAMP, 2)
assert np.allclose(np.sort(Y2, axis=0), np.sort(Y_all, axis=0)), "notebook data != acquire_next.load_training"

{'r1': 8, 'r2': 8, 'start': 32} n = 48


## 1. Audit — one unattended round re-verified end-to-end (round 2)

Round 2 was proposed by the r1 acquisition job and graded by the r2
acquisition job, both on CRC with no human in the loop (the r2 *submission*
was manual — see §6). Checks: (a) ledger HV entries 0/1/2 recomputed locally
with the read-back ref; (b) gain and stopping-rule decision; (c) calibration
z-stats recomputed from `predictions.csv` ⋈ `objectives.csv`; (d) the
committed r2 designs match the round record and the wave manifest
(SHA + seed); (e) **re-run the r2 acquisition locally** (same 40 training
points, seed 20261008, full 9⁶ lattice) and compare designs.

In [3]:
audit = {}
idx = {"start": D.src == "start", "r1": D.src.isin(["start", "r1"]), "r2": D.src.isin(["start", "r1", "r2"])}
hv_loc = [float(hypervolume_exact(Y_all[idx[k].to_numpy()], REF)) for k in ["start", "r1", "r2"]]
hv_led = [e["hv"] for e in led["entries"]]
for k, (a, b) in enumerate(zip(hv_loc, hv_led)):
    print(f"entry {k}: local {a:.10e}  ledger {b:.10e}  rel diff {abs(a-b)/b:.1e}")
    assert abs(a - b) / b < 1e-9
audit["hv_match"] = True
gains = np.diff(hv_loc); thr = led["threshold"]
print("gains:", gains, "threshold:", thr, "-> both < thr:", bool((gains < thr).all()))
assert (gains < thr).all() and led["entries"][-1]["decision"] == "stop"
n_out = [int((~np.all(Y_all[idx[k].to_numpy()] < REF, axis=1)).sum()) for k in ["start", "r1", "r2"]]
assert n_out == [e["n_outside_ref"] for e in led["entries"]], n_out
print("n outside ref box (clipped by HV, flagged):", n_out)

Z = {}
for k in ["r1", "r2"]:
    wd = os.path.join(CAMP, "waves", f"bo_C_{k}")
    p = pd.read_csv(os.path.join(wd, "predictions.csv")).merge(
        pd.read_csv(os.path.join(wd, "objectives.csv"))[["index"] + M], on="index")
    Z[k] = p
    for c in M:
        z = (p[c] - p[f"{c}_pred_mean"]) / p[f"{c}_pred_sd"]
        mine = {"mean_z": round(float(z.mean()), 3), "rms_z": round(float(np.sqrt((z**2).mean())), 3),
                "cov95": int((z.abs() <= 1.96).sum())}
        assert mine == recs[k[1]]["calibration"][c], (k, c, mine)
print("calibration z-stats: recomputed == round_records for r1, r2")

man = json.load(open(os.path.join(CAMP, "waves", "bo_C_r2", "manifest.json")))
dm2 = D[D.src == "r2"][[f"{t}_omega" for t in T]].to_numpy(float)
assert np.allclose(dm2, np.asarray(recs["2"]["designs"]), atol=1e-12)
mx = man.get("manifest_extra", man)
print("manifest keys:", sorted(k for k in man if "bo" in k or "acq" in k or "weight" in k or k == "manifest_extra"))

entry 0: local 6.6062715162e+17  ledger 6.6062715162e+17  rel diff 0.0e+00
entry 1: local 6.9692420940e+17  ledger 6.9692420940e+17  rel diff 0.0e+00
entry 2: local 7.0610491747e+17  ledger 7.0610491747e+17  rel diff 0.0e+00
gains: [3.62970578e+16 9.18070807e+15] threshold: 5.262996664950399e+16 -> both < thr: True
n outside ref box (clipped by HV, flagged): [0, 2, 5]
calibration z-stats: recomputed == round_records for r1, r2
manifest keys: ['acq_seed', 'bo_gp_sha', 'bo_round', 'sobol', 'weights']


In [4]:
def _find(d, key):
    if isinstance(d, dict):
        if key in d: return d[key]
        for v in d.values():
            r = _find(v, key)
            if r is not None: return r
    return None
assert _find(man, "bo_gp_sha") == led["bo_gp_sha"] and _find(man, "acq_seed") == 20261008
assert np.allclose(np.asarray(_find(man, "weights")), np.asarray(recs["2"]["weights"]))
print("r2 manifest: bo_gp_sha, acq_seed=20261008, weights all match round record")

# (e) local replay of the r2 acquisition
import time
from bogp.mobo.qparego import propose_qparego_batch
X40, Y40 = an.load_training(CAMP, 1)
t0 = time.time()
with gpytorch.settings.fast_pred_var():
    rb = propose_qparego_batch(X40, Y40, q=8, candidates=an.lattice_candidates(), seed=20261008)
w_ok = np.allclose(rb["weights"], np.asarray(recs["2"]["weights"]))
loc = {tuple(r) for r in np.asarray(rb["x_batch"]).round(10).tolist()}
crc = {tuple(r) for r in np.asarray(recs["2"]["designs"]).round(10).tolist()}
audit.update(replay_weights_match=bool(w_ok), replay_designs_match=len(loc & crc), replay_s=round(time.time() - t0, 1))
print(f"local r2 replay ({audit['replay_s']} s): weights match = {w_ok}; designs identical = {len(loc & crc)}/8")

r2 manifest: bo_gp_sha, acq_seed=20261008, weights all match round record


local r2 replay (361.8 s): weights match = True; designs identical = 8/8


## 2. Final front, knee candidates, designs table

In [5]:
def nondominated(Y):
    nd = np.ones(len(Y), bool)
    for i in range(len(Y)):
        nd[i] = not np.any(np.all(Y <= Y[i], axis=1) & np.any(Y < Y[i], axis=1))
    return nd
D["front"] = nondominated(Y_all)
D["front_start"] = False
D.loc[D.src == "start", "front_start"] = nondominated(Y_all[(D.src == "start").to_numpy()])
D["in_ref"] = np.all(Y_all < REF, axis=1)
F = D[D.front].copy()
print("front size:", len(F), "| by source:", F.src.value_counts().to_dict(),
      "| start-only front size:", int(D.front_start.sum()))
# start-front points knocked off by BO points
knocked = D[(D.front_start) & (~D.front)]
print("start-front points now dominated by BO picks:", list(knocked.label))

# knee candidates: min Euclidean distance to the utopia in front-normalized [0,1]^3
Yf = F[M].to_numpy(float)
lo, hi = Yf.min(0), Yf.max(0)
Fn = (Yf - lo) / (hi - lo)
F["dist_utopia"] = np.linalg.norm(Fn, axis=1)
F["knee_rank"] = F.dist_utopia.rank().astype(int)
for j, c in enumerate(M):
    F[f"{c}_norm"] = Fn[:, j]
knees = F.nsmallest(3, "dist_utopia")
print(knees[["label", "src", "dist_utopia"] + M].to_string(index=False))

front size: 26 | by source: {'start': 13, 'r2': 7, 'r1': 6} | start-only front size: 15
start-front points now dominated by BO picks: ['n0b#1', 'n0b#9']
label   src  dist_utopia  load_shed_mwh  true_curtailment_mwh  total_cost_less_synthetic_usd
n0b#8 start     0.407991    1306.585965         326405.376772                   5.631152e+08
 r1#2    r1     0.416084     835.865532         283798.848865                   5.682290e+08
 r1#6    r1     0.416460    3653.890070         270111.517634                   5.659606e+08


In [6]:
tab = F.sort_values("dist_utopia")[["label", "src", "knee_rank", "in_ref"] + [f"{t}_omega" for t in T]
                                   + [f"{t}_mw" for t in T] + ["total_mw"] + M].copy()
tab["total_cost_less_synthetic_usd"] = tab["total_cost_less_synthetic_usd"] / 1e6
tab = tab.rename(columns={"total_cost_less_synthetic_usd": "cost_ls_$M (g1-pending)"})
tab.to_csv(os.path.join(os.getcwd(), "front_designs.csv"), index=False, float_format="%.6g")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
tab.round({f"{t}_mw": 1 for t in T} | {"total_mw": 0, "load_shed_mwh": 0, "true_curtailment_mwh": 0,
           "cost_ls_$M (g1-pending)": 2})

,label,src,knee_rank,in_ref,nuclear_omega,pv_omega,tail_omega,wind_122_omega,wind_303_omega,wind_317_omega,nuclear_mw,pv_mw,tail_mw,wind_122_mw,wind_303_mw,wind_317_mw,total_mw,load_shed_mwh,true_curtailment_mwh,cost_ls_$M (g1-pending)
23,n0b#8,start,1,True,0.05000,0.05000,0.76250,0.16875,0.76250,0.88125,20.0,17.0,191.8,120.4,645.8,704.2,1699.0,1307.0,326405.0,563.12
33,r1#2,r1,2,True,0.05000,0.28750,0.88125,0.28750,0.76250,0.64375,20.0,97.9,221.6,205.1,645.8,514.4,1705.0,836.0,283799.0,568.23
37,r1#6,r1,3,True,0.05000,0.52500,0.52500,0.16875,0.64375,0.76250,20.0,178.8,132.0,120.4,545.3,609.3,1606.0,3654.0,270112.0,565.96
32,r1#1,r1,4,True,0.05000,0.64375,0.05000,0.05000,0.88125,0.52500,20.0,219.2,12.6,35.7,746.4,419.5,1453.0,6547.0,356146.0,549.20
34,r1#3,r1,5,True,0.05000,0.52500,0.05000,0.76250,0.64375,0.40625,20.0,178.8,12.6,544.0,545.3,324.6,1625.0,2781.0,276895.0,568.01
31,n0b#16,start,6,True,0.28750,0.64375,0.64375,0.40625,0.76250,0.16875,115.0,219.2,161.9,289.9,645.8,134.8,1567.0,3741.0,262823.0,573.75
47,r2#8,r2,7,True,0.05000,0.05000,1.00000,0.40625,0.88125,0.88125,20.0,17.0,251.5,289.9,746.4,704.2,2029.0,367.0,270345.0,577.99
40,r2#1,r2,8,True,0.05000,0.52500,0.76250,0.52500,0.64375,0.64375,20.0,178.8,191.8,374.6,545.3,514.4,1825.0,785.0,221749.0,580.28
45,r2#6,r2,9,True,0.05000,0.28750,1.00000,0.52500,0.52500,1.00000,20.0,97.9,251.5,374.6,444.7,799.1,1988.0,0.0,235388.0,581.79
15,n0#16,start,10,True,0.16875,0.88125,0.28750,0.64375,0.52500,0.52500,67.5,300.1,72.3,459.3,444.7,419.5,1763.0,2516.0,198793.0,582.59


In [7]:
pairs = [(0, 1), (0, 2), (1, 2)]
fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
scale = np.array([1e-3, 1e-3, 1e-6])
for ax, (a, b) in zip(axs, pairs):
    for src, col, mk in [("start", C_START, "o"), ("r1", C_R1, "s"), ("r2", C_R2, "D")]:
        s = D[D.src == src]
        ax.scatter(s[M[a]] * scale[a], s[M[b]] * scale[b], c=col, marker=mk, s=46,
                   edgecolors="white", linewidths=1.0, label={"start": "32 starting points", "r1": "BO round 1", "r2": "BO round 2"}[src], zorder=3)
    ax.scatter(F[M[a]] * scale[a], F[M[b]] * scale[b], facecolors="none", edgecolors=C_FRONT,
               s=150, linewidths=2.0, label="3-objective front (n=%d)" % len(F), zorder=4)
    ax.scatter(knees[M[a]] * scale[a], knees[M[b]] * scale[b], marker="*", c="black", s=170,
               label="knee candidates (top 3)", zorder=5)
    for (_, r), off in zip(knees.iterrows(), [(8, -14), (-38, 10), (8, 8)]):
        ax.annotate(r.label, (r[M[a]] * scale[a], r[M[b]] * scale[b]), xytext=off,
                    textcoords="offset points", fontsize=9, fontweight="bold")
    ax.set_xlabel(SHORT_K[M[a]]); ax.set_ylabel(SHORT_K[M[b]])
axs[0].legend(fontsize=8.5, frameon=False, loc="upper right")
fig.suptitle("Stage-2 scenario C: 48 evaluated designs, 3-objective front ringed (cost: g1-pending caveat)",
             fontweight="bold")
fig.savefig(os.path.join(FIG, "fig1_front.png")); fig.savefig(os.path.join(FIG, "fig1_front.pdf"))

## 3. HV trajectory vs a model-based random-lattice counterfactual

**Counterfactual (model-based — labeled as such):** the prediction GPs
(`acquire_next.PredGP`, floored, layer ii) are fit on the 32 starting points
only. Each replicate draws 2 × 8 distinct unseen lattice designs uniformly at
random and samples their objectives **jointly** from the noise-inclusive
32-point posterior; cumulative HV uses the same fixed ref. For an
apples-to-apples in-model comparison, BO's actual 16 picks are scored the
same way (sampled from the same 32-point model). The realized trajectory is
the ledger. This is a model world: it says what the 32-point model *expected*
random picks to buy, not what CRC would have returned.

In [8]:
X32, Y32 = an.load_training(CAMP, 0)
gps32 = an.fit_prediction_gps(X32, Y32)
def joint_sample(gp, Xq, n, gen):
    xq = torch.tensor((Xq - gp.mx) / gp.sx, dtype=torch.float32)
    with torch.no_grad():
        post = gp.lik(gp.model(xq))
        L = torch.linalg.cholesky(post.covariance_matrix.double() + 1e-9 * torch.eye(len(Xq), dtype=torch.float64))
        eps = torch.randn(len(Xq), n, generator=gen, dtype=torch.float64)
        s = post.mean.double()[:, None] + L @ eps
    return s.numpy().T * gp.sy + gp.my          # (n, len(Xq))

cand = an.lattice_candidates()
seen = {tuple(r) for r in X32.round(10).tolist()}
gen = torch.Generator().manual_seed(20261010); rng = np.random.default_rng(20261010)
R = 400
hv_rand = np.zeros((R, 3)); hv_bo_model = np.zeros((R, 3))
X_bo = D[D.src.isin(["r1", "r2"])][[f"{t}_omega" for t in T]].to_numpy(float)
bo_samps = np.stack([joint_sample(g, X_bo, R, gen) for g in gps32], axis=-1)   # (R,16,3)
for r in range(R):
    while True:
        pick = cand[rng.choice(len(cand), 16, replace=False)]
        if not ({tuple(x) for x in pick.round(10).tolist()} & seen): break
    ys = np.stack([joint_sample(g, pick, 1, gen)[0] for g in gps32], axis=-1)  # (16,3)
    for k, n in enumerate([0, 8, 16]):
        hv_rand[r, k] = hypervolume_exact(np.vstack([Y32, ys[:n]]), REF) if n else hv_loc[0]
        hv_bo_model[r, k] = hypervolume_exact(np.vstack([Y32, bo_samps[r, :n]]), REF) if n else hv_loc[0]
def q(a): return np.percentile(a, [10, 50, 90], axis=0)
cf = pd.DataFrame({"n": [32, 40, 48], "realized_ledger": hv_loc,
                   "random_model_p10": q(hv_rand)[0], "random_model_p50": q(hv_rand)[1], "random_model_p90": q(hv_rand)[2],
                   "bo_picks_model_p10": q(hv_bo_model)[0], "bo_picks_model_p50": q(hv_bo_model)[1], "bo_picks_model_p90": q(hv_bo_model)[2]})
cf.to_csv(os.path.join(os.getcwd(), "hv_counterfactual.csv"), index=False, float_format="%.6e")
p_beat = [(hv_bo_model[:, k] > hv_rand[:, k]).mean() for k in (1, 2)]
realized_pct = [(hv_rand[:, k] < hv_loc[k]).mean() for k in (1, 2)]
print(cf.assign(**{c: cf[c] / hv_loc[0] for c in cf.columns if c != "n"}).round(4).to_string(index=False))
print("P(BO-picks model HV > random model HV), unpaired draws: n=40 %.2f, n=48 %.2f" % tuple(p_beat))
print("realized HV percentile within random-model distribution: n=40 %.2f, n=48 %.2f" % tuple(realized_pct))

 n  realized_ledger  random_model_p10  random_model_p50  random_model_p90  bo_picks_model_p10  bo_picks_model_p50  bo_picks_model_p90
32           1.0000            1.0000            1.0000            1.0000              1.0000              1.0000              1.0000
40           1.0549            1.0182            1.0959            1.2195              1.0630              1.1580              1.3318
48           1.0688            1.0651            1.1691            1.3456              1.1288              1.2659              1.4591
P(BO-picks model HV > random model HV), unpaired draws: n=40 0.68, n=48 0.70
realized HV percentile within random-model distribution: n=40 0.30, n=48 0.11


In [9]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
n = np.array([32, 40, 48]); base = hv_loc[0]
ax.fill_between(n, q(hv_rand)[0] / base, q(hv_rand)[2] / base, color=C_START, alpha=0.25, lw=0)
ax.plot(n, q(hv_rand)[1] / base, color=C_START, lw=2, ls="--", label="random lattice picks (model-based, p10–p90)")
ax.fill_between(n, q(hv_bo_model)[0] / base, q(hv_bo_model)[2] / base, color=C_R1, alpha=0.15, lw=0)
ax.plot(n, q(hv_bo_model)[1] / base, color=C_R1, lw=2, ls=":", label="BO's picks scored by the same model (p10–p90)")
ax.plot(n, np.array(hv_loc) / base, color="black", lw=2.4, marker="o", ms=8, label="realized (ledger)")
thr_rel = thr / base
for k in (1, 2):
    ax.annotate(f"gain {gains[k-1]/base*100:+.1f}%\n(thr {thr_rel*100:.1f}%)", (n[k], hv_loc[k] / base),
                xytext=(-58, 10), textcoords="offset points", fontsize=9, fontweight="bold")
ax.set_xticks(n); ax.set_xlabel("Cumulative evaluations"); ax.set_ylabel("HV / HV(32 starting points)")
ax.legend(frameon=False, fontsize=8.5, loc="upper left")
ax.set_title("Hypervolume trajectory (fixed ref; stop rule fired after r2)")
fig.savefig(os.path.join(FIG, "fig2_hv_trajectory.png")); fig.savefig(os.path.join(FIG, "fig2_hv_trajectory.pdf"))

### 3b. Report-only: how much of the BO budget the fixed ref box could credit

The ref is pinned (nadir of the 32 starts + 0.1·span) and the stop rule used
it as pre-registered — nothing below changes the decision. But the
acquisition normalizes over the *queried* min–max, so it is free to chase
front extremes the ref box cannot see. Points outside the box add zero HV.

In [10]:
bo = D[D.src.isin(["r1", "r2"])]
out = bo[~bo.in_ref]
print(f"BO evaluations outside the fixed ref box: {len(out)}/16; of those on the final front: {int(out.front.sum())}")
print(out[["label"] + M + ["front"]].to_string(index=False))
viol = pd.DataFrame((out[M].to_numpy() > REF), columns=M, index=out.label)
print("which coordinate exceeds the ref:\n", viol)
# alt ref (REPORT-ONLY): nadir of all 48 + 0.1*span of all 48
lo48, hi48 = Y_all.min(0), Y_all.max(0); REF48 = hi48 + 0.1 * (hi48 - lo48)
hv48 = [float(hypervolume_exact(Y_all[idx[k].to_numpy()], REF48)) for k in ["start", "r1", "r2"]]
g48 = np.diff(hv48) / hv48[0] * 100
print("alt ref (48-pt nadir+10%%) — report only: gains %% of HV0 = %.2f, %.2f   (fixed-ref: %.2f, %.2f; threshold %.2f%%)"
      % (g48[0], g48[1], *(gains / hv_loc[0] * 100), thr / hv_loc[0] * 100))
alt_ref = {"ref48": REF48.tolist(), "gain_pct": g48.tolist(), "n_out_fixed": int(len(out)), "n_out_on_front": int(out.front.sum())}

BO evaluations outside the fixed ref box: 5/16; of those on the final front: 5
label  load_shed_mwh  true_curtailment_mwh  total_cost_less_synthetic_usd  front
 r1#5       0.000000          4.844409e+04                   6.684818e+08   True
 r1#7     635.518101          5.840642e+04                   6.580745e+08   True
 r2#2       0.000000          5.455751e+04                   6.666714e+08   True
 r2#3   29181.754986          1.263747e+06                   5.100661e+08   True
 r2#5    6705.233601          7.829805e+05                   5.359862e+08   True
which coordinate exceeds the ref:
        load_shed_mwh  true_curtailment_mwh  total_cost_less_synthetic_usd
label                                                                    
r1#5           False                 False                           True
r1#7           False                 False                           True
r2#2           False                 False                           True
r2#3            True          

**Stage-1 replay context** (`analysis/bo_replay/REPLAY.md` §3–4): on the
2-D, 81-cell scenario-C contour, q-ParEGO (q = 8) moved HV recovery
0.813 → 0.899 → 0.941 over two rounds vs random 0.807 → 0.885 → 0.923 —
BO ahead but by a small margin at C, because with most cells non-dominated
the M-objective HV behaves like a coverage count. Stage 2's realized
per-round gains (§3 above) should be read against that: the same
architecture, a 9⁶ = 531,441-design space, and a stop threshold set by the
shed noise floor.

## 4. Calibration verdict (pinned statistic)

In [11]:
rows, zall = [], {c: [] for c in M}
for k in ["r1", "r2"]:
    p = Z[k]
    for c in M:
        z = ((p[c] - p[f"{c}_pred_mean"]) / p[f"{c}_pred_sd"]).to_numpy()
        zall[c].append(z)
        rows.append({"round": k, "objective": c, "mean_z": z.mean(), "rms_z": np.sqrt((z**2).mean()),
                     "cov95": f"{int((np.abs(z) <= 1.96).sum())}/8"})
for c in M:
    z = np.concatenate(zall[c])
    rms, cov = np.sqrt((z**2).mean()), (np.abs(z) <= 1.96).mean()
    rows.append({"round": "pooled", "objective": c, "mean_z": z.mean(), "rms_z": rms,
                 "cov95": f"{int((np.abs(z) <= 1.96).sum())}/{len(z)} ({cov:.0%})",
                 "calibrated": bool(0.7 <= rms <= 1.3 and cov >= 0.8)})
cal = pd.DataFrame(rows)
cal.to_csv(os.path.join(os.getcwd(), "calibration.csv"), index=False, float_format="%.4f")
cal.round(3)

,round,objective,mean_z,rms_z,cov95,calibrated
0,r1,load_shed_mwh,0.031,0.325,8/8,NaN
1,r1,true_curtailment_mwh,0.285,0.554,8/8,NaN
2,r1,total_cost_less_synthetic_usd,0.392,0.910,8/8,NaN
3,r2,load_shed_mwh,0.475,1.880,7/8,NaN
4,r2,true_curtailment_mwh,0.924,2.485,6/8,NaN
5,r2,total_cost_less_synthetic_usd,0.051,1.010,8/8,NaN
6,pooled,load_shed_mwh,0.253,1.349,15/16 (94%),False
7,pooled,true_curtailment_mwh,0.604,1.800,14/16 (88%),False
8,pooled,total_cost_less_synthetic_usd,0.221,0.961,16/16 (100%),True


In [12]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, c in zip(axs, M):
    s = 1e-6 if "usd" in c else 1
    for k, col, mk in [("r1", C_R1, "s"), ("r2", C_R2, "D")]:
        p = Z[k]
        ax.errorbar(p[f"{c}_pred_mean"] * s, p[c] * s, xerr=1.96 * p[f"{c}_pred_sd"] * s, fmt=mk, color=col,
                    ms=7, mec="white", elinewidth=1.4, capsize=0, label=f"BO {k} (±1.96σ_pred)")
    lim = np.array(ax.get_xlim() + ax.get_ylim()); lim = [lim.min(), lim.max()]
    ax.plot(lim, lim, color=C_START, lw=1.4, ls="--"); ax.set_xlim(lim); ax.set_ylim(lim)
    zz = np.concatenate(zall[c])
    ax.set_title(f"{SHORT[c]}\npooled RMS(z) {np.sqrt((zz**2).mean()):.2f}, cov95 {(np.abs(zz)<=1.96).mean():.0%}", fontsize=10.5)
    ax.set_xlabel("Predicted (prediction GP)"); ax.set_ylabel("Realized (CRC)")
axs[0].legend(frameon=False, fontsize=8.5)
fig.savefig(os.path.join(FIG, "fig3_calibration.png")); fig.savefig(os.path.join(FIG, "fig3_calibration.pdf"))

## 5. Interior-vs-edge character of front designs (vs LOCATION v2)

Edge = a tier at a lattice bound (ω = 0.05 or 1.0). LOCATION v2's verdicts
to test against: curtailment location-sensitive in 15/15 pairs; cost
dominated by nuclear-vs-renewable (nuclear ~5–7× per MW); shed resolved in
*ordering* only — per-MW wind_303 ≈ wind_317 > wind_122 > nuclear > tail > pv.

In [13]:
OM = [f"{t}_omega" for t in T]
lat_lo, lat_hi = an.LAT[0], an.LAT[-1]
D["n_edge"] = ((np.isclose(D[OM], lat_lo)) | (np.isclose(D[OM], lat_hi))).sum(axis=1)
D["n_lo"] = np.isclose(D[OM], lat_lo).sum(axis=1); D["n_hi"] = np.isclose(D[OM], lat_hi).sum(axis=1)
grp = {"start (Sobol, 32)": D.src == "start", "BO r1 (8)": D.src == "r1", "BO r2 (8)": D.src == "r2",
       "front (all)": D.front, "front ∩ BO": D.front & D.src.isin(["r1", "r2"])}
edge = pd.DataFrame({g: {"n": int(m.sum()), "mean tiers at edge (of 6)": D.loc[m, "n_edge"].mean(),
                         "mean at 0.05": D.loc[m, "n_lo"].mean(), "mean at 1.0": D.loc[m, "n_hi"].mean(),
                         "share fully interior": (D.loc[m, "n_edge"] == 0).mean()} for g, m in grp.items()}).T
per_tier = pd.DataFrame({g: D.loc[m, OM].mean() for g, m in grp.items()}).T
per_tier.columns = T
edge_tier = pd.DataFrame({g: ((np.isclose(D.loc[m, OM], lat_lo)) | (np.isclose(D.loc[m, OM], lat_hi))).mean(axis=0)
                          for g, m in grp.items()}).T
edge_tier.columns = T
edge.to_csv(os.path.join(os.getcwd(), "edge_summary.csv"), float_format="%.3f")
display(edge.round(2)); display(per_tier.round(3).rename_axis("mean ω")); display(edge_tier.round(2).rename_axis("share at a bound"))

,n,mean tiers at edge (of 6),mean at 0.05,mean at 1.0,share fully interior
"start (Sobol, 32)",32.0,0.75,0.38,0.38,0.44
BO r1 (8),8.0,1.88,1.12,0.75,0.00
BO r2 (8),8.0,3.25,1.62,1.62,0.00
front (all),26.0,1.88,1.00,0.88,0.12
front ∩ BO,13.0,2.69,1.54,1.15,0.00


,nuclear,pv,tail,wind_122,wind_303,wind_317
mean ω,,,,,,
"start (Sobol, 32)",0.525,0.525,0.525,0.525,0.525,0.525
BO r1 (8),0.377,0.614,0.555,0.525,0.629,0.570
BO r2 (8),0.406,0.436,0.866,0.362,0.629,0.659
front (all),0.351,0.498,0.594,0.525,0.648,0.607
front ∩ BO,0.342,0.470,0.699,0.425,0.616,0.653


,nuclear,pv,tail,wind_122,wind_303,wind_317
share at a bound,,,,,,
"start (Sobol, 32)",0.12,0.12,0.12,0.12,0.12,0.12
BO r1 (8),0.88,0.12,0.38,0.50,0.00,0.00
BO r2 (8),1.00,0.38,0.75,0.25,0.50,0.38
front (all),0.58,0.23,0.38,0.27,0.19,0.23
front ∩ BO,1.00,0.23,0.62,0.38,0.23,0.23


In [14]:
# shed-ordering check on the 48 points: Spearman of each tier's MW with shed (and with the other objectives)
from scipy.stats import spearmanr
corr = pd.DataFrame({c: {t: spearmanr(D[f"{t}_mw"], D[c]).statistic for t in T} for c in M})
corr.index.name = "Spearman(tier MW, objective), n=48"
display(corr.round(2))
# extremes of the front (per-objective best)
ext = pd.concat([F.loc[[F[c].idxmin()]].assign(best_in=c) for c in M])
display(ext[["best_in", "label", "src"] + OM + M].round(4))

,load_shed_mwh,true_curtailment_mwh,total_cost_less_synthetic_usd
"Spearman(tier MW, objective), n=48",,,
nuclear,-0.31,-0.62,0.89
pv,-0.19,-0.42,0.44
tail,-0.16,-0.02,0.03
wind_122,-0.58,-0.48,0.51
wind_303,-0.48,-0.51,0.23
wind_317,-0.50,-0.19,0.20


,best_in,label,src,nuclear_omega,pv_omega,tail_omega,wind_122_omega,wind_303_omega,wind_317_omega,load_shed_mwh,true_curtailment_mwh,total_cost_less_synthetic_usd
25,load_shed_mwh,n0b#10,start,0.7625,0.1688,0.1688,0.6438,1.000,1.0000,0.000,1.304693e+05,6.417819e+08
36,true_curtailment_mwh,r1#5,r1,1.0000,0.7625,0.6438,1.0000,0.525,0.6438,0.000,4.844409e+04,6.684818e+08
42,total_cost_less_synthetic_usd,r2#3,r2,0.0500,0.0500,1.0000,0.0500,0.050,0.0500,29181.755,1.263747e+06,5.100661e+08


In [15]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
xs = np.arange(len(T))
for _, r in D[~D.front].iterrows():
    ax.plot(xs, r[OM].to_numpy(float), color=C_START, alpha=0.18, lw=1)
for _, r in F.iterrows():
    col = {"start": "#52514e", "r1": C_R1, "r2": C_R2}[r.src]
    ax.plot(xs, r[OM].to_numpy(float), color=col, lw=2, alpha=0.9, marker="o", ms=5)
for _, r in knees.iterrows():
    ax.plot(xs, r[OM].to_numpy(float), color="black", lw=3, ls="--")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], color=C_START, alpha=.5, label="dominated (48-pt set)"),
                   Line2D([], [], color="#52514e", lw=2, label="front — starting point"),
                   Line2D([], [], color=C_R1, lw=2, label="front — BO r1"), Line2D([], [], color=C_R2, lw=2, label="front — BO r2"),
                   Line2D([], [], color="black", lw=3, ls="--", label="knee candidates")],
          frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax.set_xticks(xs); ax.set_xticklabels(T); ax.set_yticks(an.LAT); ax.set_yticklabels([f"{v:.3g}" for v in an.LAT], fontsize=8)
ax.set_ylabel("ω (lattice level)"); ax.set_xlabel("Tier")
ax.set_title("Front designs across the 6 tiers (edge = 0.05 or 1.0)")
fig.savefig(os.path.join(FIG, "fig4_front_designs.png")); fig.savefig(os.path.join(FIG, "fig4_front_designs.pdf"))

## 6. Summary numbers (consumed by BO_STAGE2.md)

In [16]:
summary = {
    "hv": hv_loc, "gains": gains.tolist(), "threshold": thr,
    "gain_pct_of_hv0": (gains / hv_loc[0] * 100).tolist(), "threshold_pct": thr / hv_loc[0] * 100,
    "front_size": int(len(F)), "front_by_src": F.src.value_counts().to_dict(),
    "start_front_size": int(D.front_start.sum()), "knocked_off": list(knocked.label),
    "knees": knees[["label", "src"] + M].to_dict("records"),
    "calibration_pooled": cal[cal["round"] == "pooled"].drop(columns="round").to_dict("records"),
    "p_bo_model_beats_random": p_beat, "realized_pct_in_random_model": realized_pct,
    "audit": audit, "alt_ref_report_only": alt_ref,
}
json.dump(summary, open(os.path.join(os.getcwd(), "t2_summary.json"), "w"), indent=2, default=float)
print(json.dumps(summary, indent=1, default=float))

{
 "hv": [
  6.606271516247082e+17,
  6.969242093952338e+17,
  7.06104917466881e+17
 ],
 "gains": [
  3.629705777052557e+16,
  9180708071647232.0
 ],
 "threshold": 5.262996664950399e+16,
 "gain_pct_of_hv0": [
  5.4943333287556655,
  1.389695844179085
 ],
 "threshold_pct": 7.966667207072688,
 "front_size": 26,
 "front_by_src": {
  "start": 13,
  "r2": 7,
  "r1": 6
 },
 "start_front_size": 15,
 "knocked_off": [
  "n0b#1",
  "n0b#9"
 ],
 "knees": [
  {
   "label": "n0b#8",
   "src": "start",
   "load_shed_mwh": 1306.585965,
   "true_curtailment_mwh": 326405.376772,
   "total_cost_less_synthetic_usd": 563115218.28938
  },
  {
   "label": "r1#2",
   "src": "r1",
   "load_shed_mwh": 835.865532,
   "true_curtailment_mwh": 283798.848865,
   "total_cost_less_synthetic_usd": 568228961.8424821
  },
  {
   "label": "r1#6",
   "src": "r1",
   "load_shed_mwh": 3653.89007,
   "true_curtailment_mwh": 270111.51763400005,
   "total_cost_less_synthetic_usd": 565960566.163322
  }
 ],
 "calibration_pooled"